# Activity: Self-attention by hand and in code

In this activity you compute self-attention for a four-token sentence, first by hand and then with your own function:

1. Check a hand calculation of self-attention.
2. Write the attention function and look at the attention weights.

Complete every cell marked `# TODO`. After each one, run the check cell below it: it prints **Correct** when your code works.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)

## Part 1. Attention by hand

Four tokens, each with a query, a key and a value of size $d_k = 2$:

| token | Q | K | V |
|---|---|---|---|
| the | [0, 1] | [0, 1] | [0, 1] |
| dog | [1, 0] | [2, 0] | [4, 0] |
| chased | [1, 1] | [0, −1] | [0, 2] |
| it | [2, 0] | [1, 1] | [1, 1] |

For the token **"dog"**, compute on paper:

1. the four scores $q_{dog} \cdot k_j$,
2. the scores divided by $\sqrt{2}$,
3. the softmax of those values (the attention weights),
4. the output: the weighted sum of the values.

Write your results below, rounded to two decimals, and run the check.

In [ ]:
# TODO: replace each None with your hand-computed numbers (two decimals are enough).
my_scores = None    # e.g. [0, 0, 0, 0]
my_weights = None   # four numbers that add up to 1
my_output = None    # two numbers

In [ ]:
Q = np.array([[0, 1], [1, 0], [1, 1], [2, 0]], dtype=float)
K = np.array([[0, 1], [2, 0], [0, -1], [1, 1]], dtype=float)
V = np.array([[0, 1], [4, 0], [0, 2], [1, 1]], dtype=float)
tokens = ["the", "dog", "chased", "it"]


def check_hand_calculation():
    s = K @ Q[1]
    w = np.exp(s / np.sqrt(2)) / np.exp(s / np.sqrt(2)).sum()
    o = w @ V
    if my_scores is None or my_weights is None or my_output is None:
        print("Not yet: fill in my_scores, my_weights and my_output.")
        return
    for name, mine, ref in [("scores", my_scores, s), ("weights", my_weights, w), ("output", my_output, o)]:
        if np.allclose(mine, ref, atol=0.02):
            print(f"Correct: {name} = {np.round(ref, 2)}")
        else:
            print(f"Check your {name}: you wrote {mine}, expected about {np.round(ref, 2)}")


check_hand_calculation()

## Part 2. The attention function

$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right) V$$

Write the function with NumPy. It receives matrices with one row per token and returns the output (one row per token) and the matrix of attention weights.

In [ ]:
def softmax(x):
    """Softmax over the last axis. Subtracting the maximum avoids overflow."""
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def attention(Q, K, V):
    d_k = Q.shape[-1]
    # TODO: compute the scaled scores, the weights (softmax) and the output.
    weights = None
    output = None
    return output, weights

In [ ]:
def check_attention():
    out, w = attention(Q, K, V)
    if out is None or w is None:
        print("Not yet: attention() still returns None.")
        return
    ok = (w.shape == (4, 4) and np.allclose(w.sum(axis=1), 1)
          and np.isclose(w[3, 1], 0.735, atol=0.005) and np.allclose(out[3], [3.117, 0.309], atol=0.005))
    if ok:
        print("Correct: every row of weights sums to 1, and 'it' gives 0.73 of its attention to 'dog'.")
    else:
        print("Check your function: the row of 'it' should be about 0.04, 0.73, 0.04, 0.18.")


check_attention()

In [ ]:
# Visualize the attention weights: row = the token that looks, column = the token it looks at.
out, w = attention(Q, K, V)
if w is not None:
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.imshow(w, cmap="Blues", vmin=0, vmax=1)
    ax.set_xticks(range(4), tokens)
    ax.set_yticks(range(4), tokens)
    for i in range(4):
        for j in range(4):
            ax.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center")
    ax.set_title("Attention weights")
    plt.show()

In [ ]:
# Use this cell to test your predictions for question 2.
Q2 = Q.copy()
Q2[3] = [4, 0]          # the query of "it", doubled
out2, w2 = attention(Q2, K, V)
if w2 is not None:
    print("Weights of 'it' with the doubled query:", np.round(w2[3], 3))

## Questions

Answer briefly in this cell.

1. Which token does "chased" look at most? Using the queries and keys, explain why.
2. Double the query of "it" to [4, 0]. Before running the cell above, predict: does its attention to "dog" grow or shrink? Then check.
3. Without the division by $\sqrt{d_k}$, the scores of "it" would be [0, 4, 0, 2]. Compute its weight on "dog" and compare it with 0.73. What does the division do to the weights?

*Your answers:*